Goal: 3 cleaned datasets ni common column structure ki teesukoni one integrated dataset create cheyyadam.

Important: Merge/join cheyyam. Companies overlap very little. So common schema create chesi row-wise combine (concat) chestham.

First cell — Load cleaned datasets

Why: 00_data_cleaning.ipynb lo save chesina cleaned files ni use chestham.

In [1]:
# Import Pandas for data manipulation
import pandas as pd

print("Libraries imported successfully. ✅")

Libraries imported successfully. ✅


In [2]:
# Load the cleaned datasets

funding_df = pd.read_csv(
    "../data/processed/funding_cleaned.csv"
)

india_df = pd.read_csv(
    "../data/processed/india_ai_startups_cleaned.csv"
)

unicorn_df = pd.read_csv(
    "../data/processed/unicorn_cleaned.csv"
)

print("All cleaned datasets loaded successfully. ✅")

All cleaned datasets loaded successfully. ✅


Check dataset sizes

Why: Confirm that all cleaned records are available before integration.

In [3]:
# Check dataset sizes

print("Funding Dataset:", funding_df.shape)
print("India AI Dataset:", india_df.shape)
print("Unicorn Dataset:", unicorn_df.shape)

Funding Dataset: (38, 27)
India AI Dataset: (112, 22)
Unicorn Dataset: (1233, 7)


Check original columns

Why: We need to know the exact source columns before mapping them into a common structure.

In [4]:
# Check the original columns

print("========== FUNDING DATASET ==========")
print(funding_df.columns.tolist())

print("\n========== INDIA AI DATASET ==========")
print(india_df.columns.tolist())

print("\n========== UNICORN DATASET ==========")
print(unicorn_df.columns.tolist())

========== FUNDING DATASET ==========
['company', 'deal_date', 'round_type', 'amount_usd_millions', 'pre_money_valuation_millions', 'post_money_valuation_millions', 'lead_investors', 'other_investors', 'sector', 'subsector', 'hq_city', 'hq_country', 'founded_year', 'ceo', 'employees_approx', 'annual_revenue_millions', 'profitable', 'ipo_status', 'key_products', 'open_source', 'source', 'year', 'quarter', 'month', 'deal_size_tier', 'is_us_company', 'stage_number']

========== INDIA AI DATASET ==========
['company', 'website', 'founded_year', 'city', 'state', 'stage', 'ai_type', 'sector', 'sub_sector', 'total_funding_usd', 'latest_round', 'latest_round_amt_usd', 'latest_round_year', 'founders', 'lead_investors', 'employees_range', 'is_unicorn', 'valuation_usd', 'govt_backed', 'iit_founded', 'yc_backed', 'description']

========== UNICORN DATASET ==========
['Unnamed: 0', 'Company', 'Valuation ($B)', 'Date Joined', 'Country', 'City', 'Industry']


Create Funding Dataset Mapping

Why: The three datasets use different column names. We convert them into one common structure.

In [5]:
# Create the standardized Funding dataset

funding_standard = pd.DataFrame({
    "company": funding_df["company"],
    "founded_year": funding_df["founded_year"],
    "sector": funding_df["sector"],
    "subsector": funding_df["subsector"],
    "country": funding_df["hq_country"],
    "city": funding_df["hq_city"],
    "funding_usd_millions": funding_df["amount_usd_millions"],
    "valuation_usd_millions": funding_df["post_money_valuation_millions"],
    "employees": funding_df["employees_approx"],
    "startup_stage": funding_df["round_type"],
    "profitable": funding_df["profitable"],
    "is_unicorn": None,
    "data_source": "Funding Dataset"
})

print("Funding dataset standardized. ✅")
print(funding_standard.shape)

Funding dataset standardized. ✅
(38, 13)


Create India AI Dataset Mapping

Why: Convert India AI column names into the same structure.

In [6]:
# Create the standardized India AI dataset

india_standard = pd.DataFrame({
    "company": india_df["company"],
    "founded_year": india_df["founded_year"],
    "sector": india_df["sector"],
    "subsector": india_df["sub_sector"],
    "country": "India",
    "city": india_df["city"],
    "funding_usd_millions": india_df["total_funding_usd"] / 1_000_000,
    "valuation_usd_millions": india_df["valuation_usd"] / 1_000_000,
    "employees": india_df["employees_range"],
    "startup_stage": india_df["stage"],
    "profitable": None,
    "is_unicorn": india_df["is_unicorn"],
    "data_source": "India AI Dataset"
})

print("India AI dataset standardized. ✅")
print(india_standard.shape)

India AI dataset standardized. ✅
(112, 13)


Create Unicorn Dataset Mapping

Why: The Unicorn dataset has fewer features, so we map only the fields that are actually available. Missing features remain empty rather than being invented.

In [7]:
# Create the standardized Unicorn dataset

unicorn_standard = pd.DataFrame({
    "company": unicorn_df["Company"],
    "founded_year": pd.to_datetime(
        unicorn_df["Date Joined"],
        errors="coerce"
    ).dt.year,
    "sector": unicorn_df["Industry"],
    "subsector": None,
    "country": unicorn_df["Country"],
    "city": unicorn_df["City"],
    "funding_usd_millions": None,
    "valuation_usd_millions": unicorn_df["Valuation ($B)"] * 1000,
    "employees": None,
    "startup_stage": "Unicorn",
    "profitable": None,
    "is_unicorn": True,
    "data_source": "Unicorn Dataset"
})

print("Unicorn dataset standardized. ✅")
print(unicorn_standard.shape)

Unicorn dataset standardized. ✅
(1233, 13)


Check standardized columns

Why: Before combining anything, all three datasets must have exactly the same columns.

In [8]:
# Check standardized columns

print("Funding columns:", funding_standard.columns.tolist())

print("\nIndia AI columns:", india_standard.columns.tolist())

print("\nUnicorn columns:", unicorn_standard.columns.tolist())

Funding columns: ['company', 'founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'profitable', 'is_unicorn', 'data_source']

India AI columns: ['company', 'founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'profitable', 'is_unicorn', 'data_source']

Unicorn columns: ['company', 'founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'profitable', 'is_unicorn', 'data_source']


Verify column consistency

Why: This prevents the column mismatch errors we had earlier.

In [9]:
# Verify that all standardized datasets have the same columns

print(
    "Funding == India AI:",
    list(funding_standard.columns) == list(india_standard.columns)
)

print(
    "India AI == Unicorn:",
    list(india_standard.columns) == list(unicorn_standard.columns)
)

Funding == India AI: True
India AI == Unicorn: True


Combine the datasets

Why: We use concat() because the datasets represent different startup records. We are stacking records, not incorrectly joining companies.

In [10]:
# Combine all standardized datasets row by row

integrated_df = pd.concat(
    [
        funding_standard,
        india_standard,
        unicorn_standard
    ],
    ignore_index=True
)

print("Datasets integrated successfully. ✅")
print("Integrated dataset shape:", integrated_df.shape)

Datasets integrated successfully. ✅
Integrated dataset shape: (1383, 13)


C:\Users\Rahul\AppData\Local\Temp\ipykernel_23068\1316571520.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  integrated_df = pd.concat(


Create company key

Why: A standardized company key helps us compare company names across datasets later.

In [11]:
# Create a standardized company key

integrated_df["company_key"] = (
    integrated_df["company"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Company keys created successfully. ✅")

Company keys created successfully. ✅


Check company overlap

Why: We want to understand whether the same companies appear across different datasets.

In [12]:
# Create a standard company key for each dataset

funding_standard["company_key"] = (
    funding_standard["company"]
    .astype(str)
    .str.lower()
    .str.strip()
)

india_standard["company_key"] = (
    india_standard["company"]
    .astype(str)
    .str.lower()
    .str.strip()
)

unicorn_standard["company_key"] = (
    unicorn_standard["company"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Company keys created successfully. ✅")

Company keys created successfully. ✅


Create company sets

Why: Convert company names into sets so we can easily find overlaps.

In [13]:
# Create company sets

funding_companies = set(funding_standard["company_key"])
india_companies = set(india_standard["company_key"])
unicorn_companies = set(unicorn_standard["company_key"])

print("Company sets created successfully. ✅")

Company sets created successfully. ✅


Find company overlaps

Why: Check companies appearing in multiple datasets.

In [14]:
# Find overlapping companies

funding_india = funding_companies & india_companies
funding_unicorn = funding_companies & unicorn_companies
india_unicorn = india_companies & unicorn_companies
all_three = funding_companies & india_companies & unicorn_companies

print("Funding ∩ India AI:", len(funding_india))
print("Funding ∩ Unicorn:", len(funding_unicorn))
print("India AI ∩ Unicorn:", len(india_unicorn))
print("All three:", len(all_three))

Funding ∩ India AI: 0
Funding ∩ Unicorn: 11
India AI ∩ Unicorn: 6
All three: 0


Calculate overlaps

Why: This shows how much company information overlaps between the datasets.

In [15]:
# Create company sets for overlap analysis

funding_companies = set(
    funding_standard["company_key"]
)

india_companies = set(
    india_standard["company_key"]
)

unicorn_companies = set(
    unicorn_standard["company_key"]
)

print("Funding companies:", len(funding_companies))
print("India AI companies:", len(india_companies))
print("Unicorn companies:", len(unicorn_companies))

Funding companies: 28
India AI companies: 112
Unicorn companies: 1229


Display Overlapping Companies

Why: See exactly which companies overlap.

In [16]:
# Display overlapping companies

print("Funding ∩ India AI:", sorted(funding_india))
print("Funding ∩ Unicorn:", sorted(funding_unicorn))
print("India AI ∩ Unicorn:", sorted(india_unicorn))
print("All three:", sorted(all_three))

Funding ∩ India AI: []
Funding ∩ Unicorn: ['anthropic', 'cohere', 'coreweave', 'databricks', 'hugging face', 'inflection ai', 'mistral ai', 'openai', 'scale ai', 'stability ai', 'together ai']
India AI ∩ Unicorn: ['darwinbox', 'fractal analytics', 'razorpay', 'turing', 'uniphore', 'zepto']
All three: []


Combine datasets

Why: Combine all standardized records into one dataset.

In [17]:
# Combine all standardized datasets

integrated_df = pd.concat(
    [
        funding_standard,
        india_standard,
        unicorn_standard
    ],
    ignore_index=True
)

print("Datasets integrated successfully. ✅")
print("Shape:", integrated_df.shape)

Datasets integrated successfully. ✅
Shape: (1383, 14)


C:\Users\Rahul\AppData\Local\Temp\ipykernel_23068\4119140294.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  integrated_df = pd.concat(


Check columns

Why: Confirm all 14 standardized columns exist.

In [18]:
# Check integrated columns

print("Number of columns:", len(integrated_df.columns))
print(integrated_df.columns.tolist())

Number of columns: 14
['company', 'founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'profitable', 'is_unicorn', 'data_source', 'company_key']


Check data sources

Why: Confirm how many records came from each dataset.

In [19]:
# Check data source distribution

print(
    integrated_df["data_source"].value_counts()
)

data_source
Unicorn Dataset     1233
India AI Dataset     112
Funding Dataset       38
Name: count, dtype: int64


Check missing values

Why: Identify missing information before feature engineering.

In [20]:
# Check missing values

print(
    integrated_df.isnull().sum()
)

company                      0
founded_year                 0
sector                       0
subsector                 1233
country                      0
city                         0
funding_usd_millions      1233
valuation_usd_millions       7
employees                 1233
startup_stage                0
profitable                1345
is_unicorn                  38
data_source                  0
company_key                  0
dtype: int64


Check duplicate rows

Why: Make sure there are no exact duplicate records.

In [21]:
# Check duplicate rows

print(
    "Duplicate rows:",
    integrated_df.duplicated().sum()
)

Duplicate rows: 0


Save integrated dataset

Why: Save the final integrated dataset for the next notebooks.

In [22]:
# Save the integrated dataset

integrated_df.to_csv(
    "../data/processed/integrated_startup_data.csv",
    index=False
)

print("Integrated dataset saved successfully. ✅")

Integrated dataset saved successfully. ✅


Final verification

Why: Confirm the saved file is correct.

In [23]:
# Verify the saved dataset

check_df = pd.read_csv(
    "../data/processed/integrated_startup_data.csv"
)

print("Final shape:", check_df.shape)

Final shape:

 (1383, 14)
